# 04 — Causal Forest

**Key advantage over meta-learners:** CausalForestDML provides confidence intervals on CATE,
enabling statements like "this segment's uplift is significantly positive" rather than just point estimates.

**Honest splitting:** each tree is fit on a subset of the data; estimates are evaluated on held-out leaves,
preventing overfitting of the CATE surface.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_data, split, get_Xyt, FEATURE_COLS
from src.metrics import evaluate_all, qini_curve, qini_coefficient, uplift_by_decile

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

In [ ]:
df = load_data(percent10=True)
train, val, test = split(df)
X_tr, y_tr, t_tr = get_Xyt(train)
X_te, y_te, t_te = get_Xyt(test)

## Causal Forest via EconML CausalForestDML

In [ ]:
from econml.dml import CausalForestDML
from lightgbm import LGBMClassifier, LGBMRegressor
import time

# CausalForestDML: residualizes Y and T with cross-fitted nuisances,
# then fits a forest on the residual pseudo-outcome with honest splitting.
cf = CausalForestDML(
    model_y=LGBMClassifier(n_estimators=200, num_leaves=31, n_jobs=-1, verbose=-1, random_state=42),
    model_t=LGBMClassifier(n_estimators=200, num_leaves=31, n_jobs=-1, verbose=-1, random_state=42),
    n_estimators=500,
    min_samples_leaf=50,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    cv=5,
)

print('Fitting CausalForestDML...')
t0 = time.time()
cf.fit(y_tr.reshape(-1, 1), t_tr, X=X_tr)
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# Point estimates and confidence intervals
tau_cf = cf.effect(X_te).flatten()
tau_ci = cf.effect_interval(X_te, alpha=0.10)  # 90% CI
ci_lo, ci_hi = tau_ci[0].flatten(), tau_ci[1].flatten()

print('Causal Forest metrics:')
print(evaluate_all(y_te, tau_cf, t_te, 'Causal Forest'))

## CATE distribution with confidence intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: histogram of CATE estimates
ax = axes[0]
ax.hist(tau_cf, bins=60, color='#2ca02c', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', linestyle='--', label='τ=0')
ax.axvline(tau_cf.mean(), color='blue', linestyle='-', label=f'Mean τ = {tau_cf.mean():.4f}')
ax.set_xlabel('Predicted CATE τ̂(x)')
ax.set_ylabel('Count')
ax.set_title('Distribution of individual treatment effect estimates\n(Causal Forest, test set)')
ax.legend(fontsize=9)

# Right: sort by tau, plot CI bands for top-500
ax2 = axes[1]
top_idx = np.argsort(tau_cf)[::-1][:500]
rank = np.arange(500)
ax2.fill_between(rank, ci_lo[top_idx], ci_hi[top_idx], alpha=0.25, color='green', label='90% CI')
ax2.plot(rank, tau_cf[top_idx], color='green', linewidth=1.2, label='Point estimate')
ax2.axhline(0, color='red', linestyle='--', linewidth=0.8)
ax2.set_xlabel('Users ranked by predicted uplift (top 500)')
ax2.set_ylabel('Estimated CATE τ̂(x)')
ax2.set_title('CATE point estimates + 90% CI\n(honest CI via honest splitting)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../reports/causal_forest_cate.png', bbox_inches='tight')
plt.show()

## Variable importance (heterogeneity drivers)

In [ ]:
importances = cf.feature_importances_
imp_series = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(imp_series.index, imp_series.values, color='#9467bd')
ax.set_ylabel('Feature importance (causal forest)')
ax.set_title('Heterogeneity drivers — which features explain variation in uplift?')
plt.tight_layout()
plt.savefig('../reports/causal_forest_importance.png', bbox_inches='tight')
plt.show()
print(imp_series.round(4))

## Uplift by decile and Qini curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Decile chart
bins, realized = uplift_by_decile(y_te, tau_cf, t_te)
colors_d = ['#d62728' if v < 0 else '#1f77b4' for v in realized]
axes[0].bar(bins, realized * 100, color=colors_d)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Decile (1=highest predicted uplift)')
axes[0].set_ylabel('Realized uplift (%)')
axes[0].set_title('Uplift by decile — Causal Forest')

# Qini
x, q = qini_curve(y_te, tau_cf, t_te)
frac = x / len(y_te)
axes[1].plot(frac, q, color='#9467bd', linewidth=2,
             label=f'Causal Forest  Qini={qini_coefficient(y_te, tau_cf, t_te):.3f}')
axes[1].plot([0, 1], [0, q[-1]], 'k--', linewidth=0.8, label='Random')
axes[1].set_xlabel('Fraction targeted')
axes[1].set_ylabel('Qini score')
axes[1].set_title('Qini curve — Causal Forest')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../reports/causal_forest_qini.png', bbox_inches='tight')
plt.show()

In [ ]:
# Save for app
import pickle
with open('../data/meta_learner_predictions.pkl', 'rb') as f:
    saved = pickle.load(f)

saved['predictions']['Causal Forest'] = tau_cf
with open('../data/meta_learner_predictions.pkl', 'wb') as f:
    pickle.dump(saved, f)
print('Updated predictions with Causal Forest.')